# Lesson 1: graph states and cluster states

**The one-sentence version of MBQC:** first make a big, fixed, entangled state; then compute by
measuring single qubits, one at a time, choosing each measurement basis based on earlier
outcomes. No two-qubit gates during the computation. All the entangling happens up front.
([Raussendorf & Briegel, arXiv:quant-ph/0010033](https://arxiv.org/abs/quant-ph/0010033))

The entangled resource is a **graph state**. Take any graph $G$ (vertices = qubits, edges = who gets entangled):

1. put every qubit in $|+\rangle = (|0\rangle + |1\rangle)/\sqrt2$;
2. apply a controlled-Z (CZ) on every edge.

$$|G\rangle = \prod_{(a,b)\in E} CZ_{ab}\, |+\rangle^{\otimes V}.$$

CZ gates commute with each other, so the order doesn't matter. A **cluster state** is the graph state of a
line (1D) or a square lattice (2D).

Everything in this course uses a tiny simulator in `mbqc/state.py`: a state of $n$ qubits is a NumPy array of shape
`(2, 2, ..., 2)` with one axis per qubit, and each axis has a label. Read that file once; it's about 150 lines.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from mbqc import *

rng = np.random.default_rng(2026)

## Building a cluster state

`graph_state(nodes, edges)` does exactly steps 1 and 2. CZ is just a sign flip on amplitudes where both qubits are 1.

In [2]:
n = 5
cluster = graph_state(range(n), line_edges(n))
print("labels:", cluster.labels)
# All amplitudes have equal size; the sign is (-1)^(number of edges with both ends = 1)
v = cluster.vector()
for k in [0b00000, 0b00011, 0b00111, 0b01010]:
    print(f"{k:05b}  amplitude * sqrt(32) = {v[k].real * np.sqrt(32):+.0f}")

labels: [0, 1, 2, 3, 4]
00000  amplitude * sqrt(32) = +1
00011  amplitude * sqrt(32) = -1
00111  amplitude * sqrt(32) = +1
01010  amplitude * sqrt(32) = +1


## Stabilizers: the graph state in one line per qubit

Because $CZ\,(X\otimes I)\,CZ = X\otimes Z$, the operator $X_a$ that stabilizes $|+\rangle_a$ turns into
$$K_a = X_a \prod_{b\in N(a)} Z_b$$
after the CZs. So $K_a |G\rangle = |G\rangle$ for every vertex $a$, and these $n$ commuting operators pin down
$|G\rangle$ uniquely. This is the language every MBQC paper uses; e.g. [Hein et al., arXiv:quant-ph/0602096](https://arxiv.org/abs/quant-ph/0602096).

In [3]:
edges = line_edges(n)
for a in range(n):
    K = stabilizer(edges, a)
    name = " ".join(f"{'X' if np.allclose(U, X) else 'Z'}{q}" for q, U in sorted(K.items()))
    print(f"K_{a} = {name:12s}  <K> = {cluster.expval(K).real:+.3f}")

print("not a stabilizer:  <X0 X1> =", round(cluster.expval({0: X, 1: X}).real, 3))

K_0 = X0 Z1         <K> = +1.000
K_1 = Z0 X1 Z2      <K> = +1.000
K_2 = Z1 X2 Z3      <K> = +1.000
K_3 = Z2 X3 Z4      <K> = +1.000
K_4 = Z3 X4         <K> = +1.000
not a stabilizer:  <X0 X1> = 0.0


## A 2D cluster, and how entangled it is

For a graph state, the entanglement entropy of a region $A$ is a simple counting problem: it equals the rank
(over bits, i.e. mod 2) of the block of the adjacency matrix that connects $A$ to the rest. For a 1D cluster
every cut gives exactly 1 bit. A 2D cluster has much more entanglement across a cut, and that extra entanglement
is what lets two logical qubits talk (lesson 4).

In [4]:
def gf2_rank(M):
    M = M.copy() % 2
    rank = 0
    for col in range(M.shape[1]):
        rows = [r for r in range(rank, M.shape[0]) if M[r, col]]
        if not rows:
            continue
        M[[rank, rows[0]]] = M[[rows[0], rank]]
        for r in range(M.shape[0]):
            if r != rank and M[r, col]:
                M[r] ^= M[rank]
        rank += 1
    return rank


def cut_rank(nodes, edges, A):
    rest = [q for q in nodes if q not in A]
    M = np.zeros((len(A), len(rest)), dtype=int)
    for a, b in edges:
        for u, w in ((a, b), (b, a)):
            if u in A and w in rest:
                M[A.index(u), rest.index(w)] = 1
    return gf2_rank(M)


print("1D cluster, cuts:", [round(cluster.entropy(list(range(k))), 6) for k in range(1, n)])

nodes = [(r, c) for r in range(3) for c in range(3)]
edges2d = grid_edges(3, 3)
grid = graph_state(nodes, edges2d)
for A in ([(0, 0)], [(1, 1)], [(r, 0) for r in range(3)], [(0, 0), (0, 1), (1, 0), (1, 1)]):
    print(f"region {A}: entropy = {grid.entropy(A):.3f} bits, GF(2) cut rank = {cut_rank(nodes, edges2d, A)}")

1D cluster, cuts: [1.0, 1.0, 1.0, 1.0]
region [(0, 0)]: entropy = 1.000 bits, GF(2) cut rank = 1
region [(1, 1)]: entropy = 1.000 bits, GF(2) cut rank = 1
region [(0, 0), (1, 0), (2, 0)]: entropy = 3.000 bits, GF(2) cut rank = 3
region [(0, 0), (0, 1), (1, 0), (1, 1)]: entropy = 3.000 bits, GF(2) cut rank = 3


## Z measurements delete vertices

The first trick of MBQC: measuring a qubit in the $Z$ basis simply removes it from the graph. If the outcome is
$s$, its neighbours pick up a $Z^s$ "byproduct", which we can track and undo later. So a big 2D cluster is a
universal blank canvas: you carve out whatever graph you need with Z measurements, then compute with the rest.

Below: measure the middle of a 5-qubit line in $Z$. We get two separate 2-qubit clusters (stabilizers become $\pm1$,
with the sign set by the outcome).

In [5]:
state = graph_state(range(5), line_edges(5))
s = state.measure_z(2, rng=rng)
remaining = [(0, 1), (3, 4)]
print("outcome:", s, " qubits left:", state.labels)
for a in [0, 1, 3, 4]:
    print(f"<K_{a}> on the new graph = {state.expval(stabilizer(remaining, a)).real:+.0f}")
print("entropy between halves:", abs(round(state.entropy([0, 1]), 6)))

outcome: 1  qubits left: [0, 1, 3, 4]
<K_0> on the new graph = +1
<K_1> on the new graph = -1
<K_3> on the new graph = -1
<K_4> on the new graph = +1
entropy between halves: 0.0


**Takeaways**
* Graph state = $|+\rangle$ on vertices, CZ on edges; stabilizers $K_a = X_a\prod_{N(a)} Z_b$.
* Entanglement across a cut is a counting (rank) problem; a 1D cluster carries 1 bit per cut.
* $Z$ measurements delete vertices, up to Pauli byproducts on neighbours.

Next: what an $X$-plane measurement does. It moves a qubit's state to its neighbour and applies a gate.